# Embeddings y Búsqueda Vectorial
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

A partir del dataset ya limpio (generado en `02_Preprocesamiento.ipynb`), cubrir:

1. **Primera iteración de representación semántica** con Sentence-BERT (Sentence Transformers).
2. **Búsqueda vectorial baseline** con FAISS.
3. **Exploración de bases de datos vectoriales** (Pinecone, Chroma y alternativas) frente a FAISS.

Este notebook asume que ya ejecutaste `01_EDA.ipynb` y `02_Preprocesamiento.ipynb` al menos una vez (para que el parquet limpio exista en Google Drive).

## 0. Preparación e importaciones

Se instalan/importan las librerías necesarias y se monta Google Drive para acceder al dataset limpio guardado en el notebook anterior.

In [ ]:
# !pip install -q sentence-transformers faiss-cpu tqdm pinecone-client chromadb


In [ ]:
import os
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 1. Carga del dataset limpio

Se monta Google Drive (misma carpeta `proyecto_integrador` usada en el notebook de preprocesamiento) y se carga el parquet ya limpio, en vez de volver a descargar y limpiar el CSV crudo.

In [ ]:
from pathlib import Path

MONTAR_DRIVE = True  # cambia a False si prefieres trabajar solo con el almacenamiento efimero de Colab

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
else:
    BASE_DIR = Path("/content")

RUTA_PROCESSED = BASE_DIR / "data" / "processed"
RUTA_LIMPIO = RUTA_PROCESSED / "job_descriptions_clean.parquet"

if not RUTA_LIMPIO.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_LIMPIO}. "
        "Ejecuta primero 02_Preprocesamiento.ipynb para generarlo."
    )

df_clean = pd.read_parquet(RUTA_LIMPIO)
print(f"Registros cargados: {len(df_clean):,}")
df_clean[["Job Title", "texto_combinado"]].head(2)


## 2. Primera iteración de embeddings con Sentence-BERT

Con 1,615,940 registros, generar embeddings de **todo** el dataset en un notebook de Colab puede tardar horas incluso con GPU (según el modelo). Para esta primera iteración:

- Se trabaja sobre una **muestra representativa** (configurable, por defecto 50,000 registros) para validar el pipeline end-to-end rápidamente.
- Se usa `all-MiniLM-L6-v2`: modelo pequeño (384 dimensiones), rápido, buen punto de partida para *semantic search*. Si el desempeño no es suficiente en la fase de evaluación, se puede migrar a `all-mpnet-base-v2` (mejor calidad, más lento) en una segunda iteración.
- El embedding completo del dataset se puede ejecutar después en un job por lotes (batch) una vez validado el pipeline, idealmente con GPU.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"
TAMANO_MUESTRA_EMBEDDINGS = 50_000  # ajustar según recursos disponibles

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")
print(f"Dimensión del embedding: {modelo.get_sentence_embedding_dimension()}")


In [ ]:
muestra_embeddings = df_clean.sample(
    n=min(TAMANO_MUESTRA_EMBEDDINGS, len(df_clean)),
    random_state=42,
).reset_index(drop=True)

print(f"Registros a codificar: {len(muestra_embeddings):,}")


In [ ]:
inicio = time.time()

embeddings = modelo.encode(
    muestra_embeddings["texto_combinado"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # normaliza a norma 1 -> producto punto == similitud coseno
)

duracion = time.time() - inicio
print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total: {duracion:.1f} s  ({duracion / len(muestra_embeddings) * 1000:.2f} ms/registro)")


In [ ]:
np.save(RUTA_PROCESSED / "embeddings_muestra.npy", embeddings)
muestra_embeddings.to_parquet(RUTA_PROCESSED / "muestra_embeddings_meta.parquet", index=False)
print("Embeddings y metadatos guardados en Drive (persisten entre sesiones de Colab).")


### 2.1 Prueba rápida de búsqueda (baseline con FAISS)

Antes de explorar bases de datos vectoriales externas, se valida el pipeline con **FAISS** (tal como se planteó en el anteproyecto), usando un índice plano (`IndexFlatIP`) sobre la muestra.

In [ ]:
import faiss

dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)  # producto interno == coseno, porque los vectores están normalizados
indice_faiss.add(embeddings)

print(f"Vectores indexados en FAISS: {indice_faiss.ntotal:,}")


In [ ]:
def buscar_ofertas(consulta: str, k: int = 5):
    vector_consulta = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    similitudes, indices = indice_faiss.search(vector_consulta, k)

    resultados = muestra_embeddings.iloc[indices[0]].copy()
    resultados["similitud"] = similitudes[0]
    return resultados[["Job Title", "Role", "Country", "Work Type", "similitud"]]

# Ejemplo
buscar_ofertas("python developer with machine learning and NLP experience", k=5)


## 3. Exploración de bases de datos vectoriales

El anteproyecto planteó **FAISS** como motor de búsqueda vectorial. FAISS es una biblioteca (no una base de datos): es muy rápida y gratuita, pero corre en memoria/proceso local, no maneja persistencia ni actualizaciones incrementales de forma nativa, y no está pensada como servicio con API propia. Para la etapa de integración (API + interfaz web) conviene comparar alternativas de **bases de datos vectoriales** administradas o auto-hospedadas.

| Opción | Tipo | Costo | Persistencia / escalabilidad | Filtrado por metadatos | Curva de aprendizaje | Cuándo conviene |
|---|---|---|---|---|---|---|
| **FAISS** | Biblioteca en memoria | Gratis | Manual (hay que guardar/cargar el índice); escala vertical | Limitado, se maneja aparte en Pandas/SQL | Baja | Prototipos, datasets que caben en memoria/disco de un solo servidor |
| **Chroma** | Base de datos vectorial embebida/self-hosted | Gratis (open source) | Persistencia en disco nativa; fácil de correr local o en Docker | Sí, nativo | Baja | Buen punto medio para pasar de FAISS a algo con persistencia sin salir de Python |
| **Qdrant** | Base de datos vectorial self-hosted o cloud (tiene *free tier*) | Gratis (self-host) / plan gratuito cloud limitado | Persistencia, filtrado avanzado, buena para producción | Sí, muy completo | Media | Cuando se quiere una API HTTP/gRPC propia y filtros complejos sobre metadatos |
| **Pinecone** | Base de datos vectorial 100% administrada (SaaS) | Plan gratuito limitado (una región, límite de vectores); pago por uso a partir de cierto volumen | Totalmente administrada, escala automáticamente | Sí | Baja (API simple) | Cuando no se quiere administrar infraestructura y se prioriza velocidad de desarrollo |
| **Weaviate** | Base de datos vectorial self-hosted o cloud | Gratis (self-host) / cloud con costo | Persistencia, soporta búsqueda híbrida (vectorial + palabras clave) | Sí | Media-alta | Si además de similitud semántica se quiere combinar con búsqueda léxica (BM25) |

**Recomendación para esta iteración:** dado que el equipo ya validó FAISS en el anteproyecto y el presupuesto no contempla servicios pagos recurrentes más allá de lo presupuestado, una ruta razonable es:

1. Mantener **FAISS** como baseline de referencia (ya funciona, es gratis, cero dependencias externas).
2. Probar **Pinecone** en su capa gratuita para evaluar si la facilidad de integración (API lista, sin mantener infraestructura) compensa el límite de vectores del plan free — es útil para la demo/API del alcance del proyecto.
3. Si el plan gratuito de Pinecone resulta insuficiente para 1.6M de registros, evaluar **Qdrant** o **Chroma** self-hosted en el mismo VPS ya presupuestado (Railway), que no tienen límite artificial de vectores.

A continuación, un ejemplo mínimo (no ejecutado, requiere API key) de cómo se vería la integración con Pinecone para comparar el mismo flujo de búsqueda.

In [ ]:
# Ejemplo ilustrativo de integración con Pinecone (requiere cuenta y API key gratuitas).
# No se ejecuta aquí porque necesita credenciales.
#
# from pinecone import Pinecone, ServerlessSpec
#
# pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
#
# NOMBRE_INDICE = "ofertas-laborales"
# if NOMBRE_INDICE not in [i.name for i in pc.list_indexes()]:
#     pc.create_index(
#         name=NOMBRE_INDICE,
#         dimension=dimension,          # 384 para all-MiniLM-L6-v2
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1"),
#     )
#
# indice_pinecone = pc.Index(NOMBRE_INDICE)
#
# vectores = [
#     (str(row["Job Id"]), emb.tolist(), {"Job Title": row["Job Title"], "Country": row["Country"]})
#     for row, emb in zip(muestra_embeddings.to_dict("records"), embeddings)
# ]
# for lote in range(0, len(vectores), 100):
#     indice_pinecone.upsert(vectores[lote:lote + 100])
#
# resultado = indice_pinecone.query(
#     vector=modelo.encode("python developer with machine learning").tolist(),
#     top_k=5,
#     include_metadata=True,
# )


In [ ]:
# Ejemplo ilustrativo de integración con Chroma (sí se puede instalar localmente y probar sin API key externa).
# import chromadb
#
# cliente_chroma = chromadb.PersistentClient(path="str(RUTA_PROCESSED / "chroma_db")")
# coleccion = cliente_chroma.get_or_create_collection(name="ofertas_laborales", metadata={"hnsw:space": "cosine"})
#
# coleccion.add(
#     ids=[str(i) for i in muestra_embeddings["Job Id"]],
#     embeddings=embeddings.tolist(),
#     metadatas=muestra_embeddings[["Job Title", "Role", "Country", "Work Type"]].to_dict("records"),
# )
#
# resultados_chroma = coleccion.query(
#     query_embeddings=modelo.encode(["python developer with machine learning"]).tolist(),
#     n_results=5,
# )


## 4. Próximos pasos

1. **Decidir la base de datos vectorial** definitiva a partir de esta comparación (recomendado: probar Pinecone free tier + Chroma self-hosted, y quedarse con el que mejor equilibre costo/latencia/facilidad de integración con la API).
2. **Escalar el embedding** al dataset completo (o a un subconjunto representativo mayor) una vez elegida la base de datos, idealmente en un proceso por lotes fuera del notebook.
3. Evaluar si `all-MiniLM-L6-v2` es suficiente o si conviene migrar a un modelo más grande (`all-mpnet-base-v2`) según los resultados de la evaluación cualitativa de consultas de prueba.
4. Con la base vectorial elegida, comenzar el **Objetivo específico 4**: desarrollo de la API y la interfaz web.